In [73]:
import numpy as np
import os
import pickle
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [74]:
from ucimlrepo import fetch_ucirepo

# ---------------
# fetch dataset
heart_disease = fetch_ucirepo(id=45)

# data (as pandas dataframes)
X = heart_disease.data.features
y = heart_disease.data.targets
# ---------------

In [75]:
# •	0 → No heart disease (< 50% narrowing)
# •	1 → Mild heart disease (> 50% narrowing)
# •	2 → Moderate heart disease
# •	3 → Severe heart disease
# •	4 → Very severe heart disease

In [76]:
# Replace '?' with NaN
X = X.replace("?", np.nan)

# Convert to numeric
X = X.apply(pd.to_numeric, errors="coerce")

# Combine X and y for cleaning
df = X.copy()
df["num"] = y

# Drop missing values
df = df.dropna()

# Split again
X = df.drop("num", axis=1)
y = df["num"]

# Save feature names
feature_names = X.columns.tolist()

In [77]:
df = pd.concat([X, y], axis=1)

In [78]:
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import GradientBoostingClassifier, ExtraTreesClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=1000,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

# -----------------------------
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nUnique predictions:", np.unique(y_pred))
print("Unique labels:", np.unique(y))

# -----------------------------
# Save Model
# -----------------------------
os.makedirs("model", exist_ok=True)

with open("model/rf_model.pkl", "wb") as f:
    pickle.dump({"model": model, "features": feature_names}, f)

print("\nModel saved at model/rf_model.pkl")

Accuracy: 0.5333333333333333

Classification Report:

              precision    recall  f1-score   support

           0       0.76      0.97      0.85        32
           1       0.09      0.09      0.09        11
           2       0.00      0.00      0.00         7
           3       0.00      0.00      0.00         7
           4       0.00      0.00      0.00         3

    accuracy                           0.53        60
   macro avg       0.17      0.21      0.19        60
weighted avg       0.42      0.53      0.47        60


Unique predictions: [0 1 2 3]
Unique labels: [0 1 2 3 4]

Model saved at model/rf_model.pkl
